# Teen Social Media & Mental Health — Analysis

**Question:** Is the time teenagers spend on social media associated with worse sleep, higher anxiety and a higher risk of depression? Are some platforms worse than others?

**Data:** 600 anonymised teenager records (synthetic / educational dataset). Columns: age, gender, primary platform, daily hours on social media, sleep hours, self-reported stress / anxiety / addiction / offline-interaction (all 1–10), and a binary depression label.

**TL;DR of findings** (full reasoning below):
- Teens spending **>5h/day** on social media sleep **~1.7 hours less** than peers with ≤2h/day.
- **Instagram** and **TikTok** users report the highest anxiety levels.
- Daily social-media hours is the single variable most strongly correlated with the depression label.


## 1. Setup and data loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

df = pd.read_csv("data/teen_mental_health.csv")
print(df.shape)
df.head()

## 2. Data sanity check

Always glance at types, nulls and ranges before drawing any conclusions.

In [ ]:
df.info()

In [ ]:
df.describe()

No missing values, types are correct, ranges are plausible (ages 13–18, scores 1–10). Safe to proceed.

## 3. Hypothesis 1 — More social media = less sleep

In [ ]:
heavy = df[df['daily_social_media_hours'] > 5]['sleep_hours'].mean()
light = df[df['daily_social_media_hours'] <= 2]['sleep_hours'].mean()
print(f"Average sleep, heavy users (>5h SM): {heavy:.2f} h")
print(f"Average sleep, light users (<=2h SM): {light:.2f} h")
print(f"Difference: {light - heavy:.2f} h")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=df, x="daily_social_media_hours", y="sleep_hours",
    hue="depression_label", palette={0: "#4C9AFF", 1: "#FF6B6B"},
    alpha=0.7, s=70, ax=ax
)
ax.set_title("Social media hours vs sleep (coloured by depression label)")
ax.set_xlabel("Daily social media hours")
ax.set_ylabel("Sleep hours")
ax.legend(title="Depression", labels=["No", "Yes"])
plt.show()

**Reading the chart:** the cloud slopes downward — more social media, less sleep. Red dots (depressed teens) concentrate in the bottom-right corner (high social-media use AND low sleep). The two effects appear to compound.

## 4. Hypothesis 2 — Some platforms are worse than others

In [ ]:
platform_stats = df.groupby('platform_usage').agg(
    avg_hours=('daily_social_media_hours', 'mean'),
    avg_sleep=('sleep_hours', 'mean'),
    avg_anxiety=('anxiety_level', 'mean'),
    depression_rate=('depression_label', 'mean'),
    users=('platform_usage', 'count')
).sort_values('avg_anxiety', ascending=False).round(2)

platform_stats

In [ ]:
plat_anx = df.groupby('platform_usage')['anxiety_level'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=plat_anx.index, y=plat_anx.values, hue=plat_anx.index,
            palette="rocket_r", legend=False, ax=ax)
ax.set_title("Average anxiety level by platform")
ax.set_xlabel("Platform")
ax.set_ylabel("Average anxiety (1–10)")
for i, v in enumerate(plat_anx.values):
    ax.text(i, v + 0.05, f"{v:.1f}", ha="center", fontsize=12)
plt.show()

**Reading the chart:** Instagram and TikTok lead on anxiety. This matches the fact that their users also spend the most time per day on the app — so the platform effect may simply be a *time* effect dressed up as a platform effect. Worth flagging this in the conclusions instead of claiming Instagram causes anxiety.

## 5. What correlates most with depression?

In [ ]:
num_cols = ['daily_social_media_hours', 'sleep_hours', 'stress_level',
            'anxiety_level', 'addiction_level', 'social_interaction_level',
            'depression_label']

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation between key variables")
plt.show()

In [ ]:
df[num_cols].corr()['depression_label'].sort_values(ascending=False).round(2)

**Reading the chart:**
- **Daily social-media hours** is the single variable most strongly correlated with the depression label (+0.37).
- **Offline social interaction** is the strongest *protective* factor (-0.22).
- Stress, anxiety and addiction are all tightly correlated with each other — they likely measure overlapping things.


## 6. Conclusions and recommendations

**What the data says:**
1. Time spent on social media is associated with **less sleep**, **higher anxiety**, and **a higher rate of depression** in this sample.
2. The 5-hour mark looks like a practical threshold — past it, average sleep drops below 7 hours and the depression rate roughly doubles vs the overall sample.
3. Instagram and TikTok lead on anxiety, but their users also spend the most time on the app, so we can't disentangle platform from dose.

**What I would recommend if this were a real project for a school or NGO:**
- Communicate the **5h/day threshold** as a concrete, actionable benchmark for parents.
- Treat **offline social interaction** as a protective factor worth investing in (clubs, sports, group activities).
- Don't single out one platform — focus on total daily screen time instead.

**Caveats:** this is observational data — correlation, not causation. The label is self-reported. Sample size (600) is modest. A follow-up study would benefit from longitudinal data and clinical depression assessments.
